In [1]:
import pandas as pd 

csv_file = "intervalos vazao  esmond data psmp-gn-bw-lis-pt.geant.org to perfsonar-ankara.ulakbim.gov.tr 10-24-2023.csv"
df = pd.read_csv(csv_file)
df.head()

,Data,Intervalo,Vazao
0,28-4-2023,00:00:00 a 05:59:59,904398815.5
1,28-4-2023,06:00:00 a 11:59:59,-1.0
2,28-4-2023,12:00:00 a 17:59:59,-1.0
3,28-4-2023,18:00:00 a 23:59:59,913310854.5
4,29-4-2023,00:00:00 a 05:59:59,-1.0


Clean date and interval columns to use as timestamp

In [2]:
# Split start ("Inicio") and end ("Fim") times from interval column
df[["Inicio", "Fim"]] = df["Intervalo"].str.split(" a ", expand=True)

# Convert date and start time to datetime to create a timestamp
df["timestamp"] = pd.to_datetime(df["Data"] + " " + df["Inicio"], dayfirst=True)
# df["timestamp_fim"] = pd.to_datetime(df["Data"] + " " + df["Fim"], dayfirst=True)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.drop(columns=["Intervalo", "Data", "Inicio", "Fim"])
df.head()

,Vazao,timestamp
0,904398815.5,2023-04-28 00:00:00
1,-1.0,2023-04-28 06:00:00
2,-1.0,2023-04-28 12:00:00
3,913310854.5,2023-04-28 18:00:00
4,-1.0,2023-04-29 00:00:00


Split manually choosen longest interval

In [3]:
start = "2023-06-01 12:00:00"
end    = "2023-07-17 00:00:00"

longest_interval = df[(df["timestamp"] >= start) & (df["timestamp"] <= end)]

longest_interval.shape

(183, 2)

In [4]:
import numpy as np

longest_interval["Vazao"] = longest_interval["Vazao"].replace(-1, np.nan)

longest_interval["Vazao"] = longest_interval["Vazao"].interpolate(method="linear")
print(longest_interval.isna().sum())
longest_interval.head(20)

Vazao        0
timestamp    0
dtype: int64


/tmp/ipykernel_29155/834657600.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  longest_interval["Vazao"] = longest_interval["Vazao"].replace(-1, np.nan)
/tmp/ipykernel_29155/834657600.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  longest_interval["Vazao"] = longest_interval["Vazao"].interpolate(method="linear")


,Vazao,timestamp
110,860620011.5,2023-06-01 12:00:00
111,880017050.5,2023-06-01 18:00:00
112,901774368.0,2023-06-02 00:00:00
113,912525451.5,2023-06-02 06:00:00
114,889717314.0,2023-06-02 12:00:00
115,871363431.0,2023-06-02 18:00:00
116,874513662.0,2023-06-03 00:00:00
117,903873912.0,2023-06-03 06:00:00
118,888143963.5,2023-06-03 12:00:00
119,880284833.0,2023-06-03 18:00:00


In [9]:
missing_rates = [0.1, 0.2, 0.3, 0.4]
random_seed = 42

missing_dfs = {}
for missing_rate in missing_rates:
    missing_df = df.drop(df.sample(frac=missing_rate, random_state=random_seed).index)
    missing_dfs[missing_rate] = missing_df

missing_dfs[0.1].head(20)

,Vazao,timestamp
0,904398815.5,2023-04-28 00:00:00
1,-1.0,2023-04-28 06:00:00
3,913310854.5,2023-04-28 18:00:00
4,-1.0,2023-04-29 00:00:00
5,890767363.5,2023-04-29 06:00:00
7,905971204.0,2023-04-29 18:00:00
8,912787963.0,2023-04-30 00:00:00
9,877134889.5,2023-04-30 06:00:00
11,868222143.0,2023-04-30 18:00:00
12,894699555.0,2023-05-01 00:00:00
